# Photo Library — Compositional & Style Analysis

**Goal:** Use CLIP (from Stable Diffusion's backbone) + classical CV to analyze a sample of photos for:
- **Leading lines** — edge + Hough-line detection
- **Rule of thirds** — saliency map vs. power-point alignment
- **Artistic style** — zero-shot CLIP classification against a curated style vocabulary

No GPU required for CLIP analysis; SD image-to-image runs best on CUDA but falls back to CPU.

## 0. Install dependencies

In [ ]:
# Run once — comment out after first install
%pip install -q diffusers transformers accelerate torch torchvision \
              open-clip-torch opencv-python-headless scikit-image \
              pillow matplotlib numpy scipy tqdm ipywidgets

## 1. Imports & Config

In [ ]:
import os, glob, random
from pathlib import Path

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from tqdm.notebook import tqdm

import torch
import open_clip
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist

# ── Configuration ────────────────────────────────────────────────────────────
# Point this at your photo library root
PHOTO_DIR = Path(os.path.expanduser("~/Pictures"))   # ← edit as needed
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".heic", ".tiff", ".webp")
SAMPLE_SIZE = 30          # photos to analyse (increase for richer clustering)
THUMB_SIZE  = (512, 512)  # resize for model inputs
RANDOM_SEED = 42

# CLIP model — ViT-H/14 is the same encoder used inside SD 2.x
CLIP_MODEL_NAME = "ViT-H-14"
CLIP_PRETRAINED = "laion2b_s32b_b79k"  # best open-source checkpoint

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 2. Load sample images

In [ ]:
def collect_images(root: Path, exts=IMAGE_EXTENSIONS) -> list[Path]:
    found = []
    for ext in exts:
        found.extend(root.rglob(f"*{ext}"))
        found.extend(root.rglob(f"*{ext.upper()}"))
    return sorted(set(found))

all_paths = collect_images(PHOTO_DIR)
print(f"Found {len(all_paths)} images in {PHOTO_DIR}")

random.seed(RANDOM_SEED)
sample_paths = random.sample(all_paths, min(SAMPLE_SIZE, len(all_paths)))
print(f"Sampled {len(sample_paths)} images for analysis")

def load_rgb(path: Path, size=THUMB_SIZE) -> np.ndarray:
    img = Image.open(path).convert("RGB")
    img.thumbnail(size, Image.LANCZOS)
    return np.array(img)

# Quick preview grid
preview = [load_rgb(p) for p in sample_paths[:12]]
fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for ax, img, path in zip(axes.flat, preview, sample_paths[:12]):
    ax.imshow(img)
    ax.set_title(path.name[:20], fontsize=7)
    ax.axis("off")
plt.suptitle("Sample — first 12 images", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Compositional Analysis

### 3a. Leading Lines — Canny + Probabilistic Hough

In [ ]:
def detect_leading_lines(rgb: np.ndarray,
                          canny_lo=50, canny_hi=150,
                          hough_threshold=80,
                          min_line_len=60,
                          max_line_gap=10) -> dict:
    """
    Returns:
        lines      — list of (x1,y1,x2,y2) tuples
        edge_img   — Canny edge image
        line_img   — RGB image with lines overlaid
        score      — normalised line density 0–1
    """
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(blurred, canny_lo, canny_hi)

    raw = cv2.HoughLinesP(edges, 1, np.pi / 180,
                          threshold=hough_threshold,
                          minLineLength=min_line_len,
                          maxLineGap=max_line_gap)
    lines = [tuple(l[0]) for l in raw] if raw is not None else []

    overlay = rgb.copy()
    for x1, y1, x2, y2 in lines:
        cv2.line(overlay, (x1, y1), (x2, y2), (255, 80, 0), 2)

    h, w = rgb.shape[:2]
    score = min(len(lines) / 20, 1.0)  # 20 lines → score 1.0
    return {"lines": lines, "edge_img": edges, "line_img": overlay, "score": score}

# Visualise on first 6 samples
fig, axes = plt.subplots(6, 3, figsize=(14, 22))
for row, path in enumerate(sample_paths[:6]):
    rgb = load_rgb(path)
    result = detect_leading_lines(rgb)
    axes[row, 0].imshow(rgb);               axes[row, 0].set_title("Original", fontsize=8)
    axes[row, 1].imshow(result["edge_img"], cmap="gray")
    axes[row, 1].set_title(f"Edges", fontsize=8)
    axes[row, 2].imshow(result["line_img"])
    axes[row, 2].set_title(f"Lines detected: {len(result['lines'])}  score={result['score']:.2f}", fontsize=8)
    for ax in axes[row]: ax.axis("off")
plt.suptitle("Leading Lines Detection", fontsize=15)
plt.tight_layout()
plt.show()

### 3b. Rule of Thirds — Saliency alignment

In [ ]:
def spectral_saliency(rgb: np.ndarray) -> np.ndarray:
    """Fast spectral residual saliency map."""
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    # Resize to small for speed
    small = cv2.resize(gray, (64, 64))
    dft = np.fft.fft2(small)
    mag = np.log(np.abs(dft) + 1e-8)
    smooth_mag = cv2.GaussianBlur(mag, (3, 3), 0)
    residual = mag - smooth_mag
    saliency_small = np.abs(np.fft.ifft2(np.exp(residual + 1j * np.angle(dft)))) ** 2
    saliency = cv2.resize(saliency_small, (rgb.shape[1], rgb.shape[0]))
    saliency = cv2.GaussianBlur(saliency, (21, 21), 0)
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
    return saliency

def rule_of_thirds_score(rgb: np.ndarray) -> dict:
    """
    Power points are the 4 intersections of the thirds grid.
    Score = average saliency in 10% neighbourhood of each power point.
    Returns score [0,1] and saliency map.
    """
    h, w = rgb.shape[:2]
    saliency = spectral_saliency(rgb)

    # Thirds grid lines
    vlines = [w // 3, 2 * w // 3]
    hlines = [h // 3, 2 * h // 3]
    power_points = [(x, y) for x in vlines for y in hlines]

    radius = int(min(h, w) * 0.10)
    scores = []
    for (px, py) in power_points:
        y0, y1 = max(0, py - radius), min(h, py + radius)
        x0, x1 = max(0, px - radius), min(w, px + radius)
        scores.append(saliency[y0:y1, x0:x1].mean())

    # Complement: saliency at centre (anti-thirds)
    cy, cx = h // 2, w // 2
    centre_score = saliency[cy - radius:cy + radius, cx - radius:cx + radius].mean()

    thirds_score = np.mean(scores)
    # Penalise if centre dominates
    final_score = max(0, thirds_score - 0.5 * centre_score)
    return {"score": float(final_score),
            "power_point_scores": scores,
            "saliency": saliency,
            "power_points": power_points,
            "grid_lines": (hlines, vlines)}

def draw_thirds_overlay(rgb, result):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    h, w = rgb.shape[:2]
    hlines, vlines = result["grid_lines"]

    for ax, img, title in zip(axes,
                               [rgb, result["saliency"]],
                               ["Rule of Thirds Grid", "Saliency Map"]):
        ax.imshow(img, cmap="hot" if img.ndim == 2 else None)
        for hl in hlines: ax.axhline(hl, color="lime", lw=1, alpha=0.7)
        for vl in vlines: ax.axvline(vl, color="lime", lw=1, alpha=0.7)
        for (px, py) in result["power_points"]:
            ax.plot(px, py, "o", color="yellow", markersize=8)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    plt.suptitle(f"Rule-of-Thirds score: {result['score']:.3f}", fontsize=11)
    plt.tight_layout()
    return fig

# Show for first 4 images
for path in sample_paths[:4]:
    rgb = load_rgb(path)
    result = rule_of_thirds_score(rgb)
    fig = draw_thirds_overlay(rgb, result)
    plt.show()

## 4. Batch Compositional Scoring

In [ ]:
records = []
for path in tqdm(sample_paths, desc="Scoring composition"):
    rgb = load_rgb(path)
    ll = detect_leading_lines(rgb)
    rot = rule_of_thirds_score(rgb)
    records.append({
        "path": path,
        "name": path.name,
        "leading_lines_score": ll["score"],
        "leading_lines_count": len(ll["lines"]),
        "thirds_score": rot["score"],
    })

import pandas as pd
df = pd.DataFrame(records)
df["composition_score"] = 0.5 * df["leading_lines_score"] + 0.5 * df["thirds_score"]
df_sorted = df.sort_values("composition_score", ascending=False)
print(df_sorted[["name", "leading_lines_count", "leading_lines_score",
                 "thirds_score", "composition_score"]].to_string(index=False))

## 5. Artistic Style Identification via CLIP

We use **OpenCLIP ViT-H/14** (same encoder as Stable Diffusion 2.x) for zero-shot style classification.

In [ ]:
# ── Load CLIP model ───────────────────────────────────────────────────────────
print(f"Loading {CLIP_MODEL_NAME} on {DEVICE} …")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    CLIP_MODEL_NAME, pretrained=CLIP_PRETRAINED, device=DEVICE
)
clip_model.eval()
tokenizer = open_clip.get_tokenizer(CLIP_MODEL_NAME)
print("CLIP ready.")

In [ ]:
# ── Style vocabulary ──────────────────────────────────────────────────────────
# Each entry is a (label, list-of-prompts) pair — CLIP averages prompt embeddings.
STYLE_VOCAB = {
    "Minimalist":          ["a minimalist photograph", "minimalism, negative space, clean composition"],
    "High-Key": ["a high-key photograph with bright tones and soft shadows"],
    "Low-Key / Moody":     ["a low-key dark moody photograph", "dramatic shadows, chiaroscuro lighting"],
    "Bokeh / Shallow DOF": ["a photograph with beautiful bokeh and shallow depth of field",
                            "blurred background, subject isolation"],
    "Golden Hour":         ["a golden hour photograph", "warm sunset light, long shadows"],
    "Street Photography":  ["street photography, candid moment, urban scene"],
    "Documentary / Film":  ["documentary photography, film grain, journalistic style"],
    "Abstract":            ["abstract photograph, geometric shapes, patterns, texture"],
    "Black & White":       ["black and white photograph", "monochrome, high contrast b&w"],
    "Landscape / Nature":  ["landscape photograph, nature, scenic view"],
    "Portrait":            ["a portrait photograph of a person"],
    "Macro / Detail":      ["macro photography, extreme close-up, fine detail"],
    "Architecture":        ["architectural photography, buildings, geometry"],
    "Cinematic":           ["cinematic photograph, movie still, letterbox, dramatic framing"],
    "Ethereal / Dreamy":   ["ethereal dreamy photograph, soft light, mist, fairy-tale mood"],
}

# Pre-encode style text embeddings
style_labels = list(STYLE_VOCAB.keys())
style_text_embeddings = []

with torch.no_grad():
    for label in style_labels:
        prompts = STYLE_VOCAB[label]
        tokens = tokenizer(prompts).to(DEVICE)
        emb = clip_model.encode_text(tokens)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        style_text_embeddings.append(emb.mean(dim=0))  # average over prompts

style_matrix = torch.stack(style_text_embeddings)  # [n_styles, D]
print(f"Style matrix: {style_matrix.shape}")

In [ ]:
# ── Encode all sample images ──────────────────────────────────────────────────
image_embeddings = []

with torch.no_grad():
    for path in tqdm(sample_paths, desc="CLIP image encoding"):
        img = Image.open(path).convert("RGB")
        tensor = clip_preprocess(img).unsqueeze(0).to(DEVICE)
        emb = clip_model.encode_image(tensor)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        image_embeddings.append(emb.squeeze(0))

image_matrix = torch.stack(image_embeddings)  # [n_images, D]
print(f"Image matrix: {image_matrix.shape}")

# ── Compute similarity to each style ─────────────────────────────────────────
# Cosine similarity: [n_images, n_styles]
similarity = (image_matrix @ style_matrix.T).cpu().numpy()

# Softmax for interpretable probabilities
def softmax(x, temp=0.07):
    x = x / temp
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

probs = softmax(similarity)
top_styles = [style_labels[np.argmax(probs[i])] for i in range(len(sample_paths))]

# Add to dataframe
df["primary_style"] = top_styles
df["style_confidence"] = [probs[i].max() for i in range(len(sample_paths))]

print("\nStyle distribution:")
print(df["primary_style"].value_counts().to_string())

### 5a. Style probability heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(16, max(6, len(sample_paths) * 0.35)))
im = ax.imshow(probs, aspect="auto", cmap="YlOrRd", vmin=0, vmax=probs.max())
ax.set_xticks(range(len(style_labels)))
ax.set_xticklabels(style_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(sample_paths)))
ax.set_yticklabels([p.name[:30] for p in sample_paths], fontsize=7)
plt.colorbar(im, ax=ax, label="CLIP similarity (softmax)")
ax.set_title("Image × Style Similarity Matrix", fontsize=14)
plt.tight_layout()
plt.show()

### 5b. Top images per style

In [ ]:
def show_top_per_style(n_per_style=3, n_styles=6):
    # Pick styles that have at least one top-match image
    style_indices = np.argsort(-probs, axis=0)  # [n_images, n_styles] sorted descending
    active_styles = [s for s in style_labels if df["primary_style"].eq(s).any()][:n_styles]

    fig, axes = plt.subplots(len(active_styles), n_per_style,
                              figsize=(n_per_style * 4, len(active_styles) * 3.5))
    if len(active_styles) == 1: axes = [axes]

    for row, style in enumerate(active_styles):
        si = style_labels.index(style)
        top_img_indices = style_indices[:, si][:n_per_style]
        for col, img_idx in enumerate(top_img_indices):
            ax = axes[row][col]
            rgb = load_rgb(sample_paths[img_idx])
            ax.imshow(rgb)
            conf = probs[img_idx, si]
            ax.set_title(f"{sample_paths[img_idx].name[:18]}\nconf={conf:.2f}",
                         fontsize=7)
            ax.axis("off")
        axes[row][0].set_ylabel(style, fontsize=10, rotation=0,
                                labelpad=90, va="center")
    plt.suptitle("Top Images per Artistic Style", fontsize=14)
    plt.tight_layout()
    plt.show()

show_top_per_style(n_per_style=3, n_styles=8)

## 6. Embedding Clustering — Visual Style Map

PCA → KMeans to group images by visual similarity (independent of text labels).

In [ ]:
emb_np = image_matrix.cpu().numpy()  # [n, D]

# Reduce to 2D for visualisation
pca = PCA(n_components=2, random_state=RANDOM_SEED)
coords_2d = pca.fit_transform(emb_np)

# KMeans clusters
n_clusters = min(6, len(sample_paths))
km = KMeans(n_clusters=n_clusters, random_state=RANDOM_SEED, n_init="auto")
clusters = km.fit_predict(emb_np)

fig, ax = plt.subplots(figsize=(12, 8))
colors = plt.cm.tab10(np.linspace(0, 1, n_clusters))
for i, (x, y) in enumerate(coords_2d):
    c = clusters[i]
    ax.scatter(x, y, color=colors[c], s=80, zorder=3)
    ax.annotate(f"{sample_paths[i].name[:15]}\n{top_styles[i]}",
                (x, y), fontsize=6, ha="center", va="bottom",
                xytext=(0, 6), textcoords="offset points")

ax.set_xlabel("PC 1")
ax.set_ylabel("PC 2")
ax.set_title("Visual Embedding Space (PCA 2D, KMeans clusters)", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Stable Diffusion Interrogation (Optional — GPU recommended)

Uses `BLIP-2` or `img2prompt` (SD Interrogator) to generate descriptive captions / prompts for each image — useful for understanding SD's interpretation of your photos.

In [ ]:
# ── Optional: SD-style prompt generation via BLIP-2 ──────────────────────────
# Uncomment and run if you have a GPU with ≥8 GB VRAM.

# from transformers import Blip2Processor, Blip2ForConditionalGeneration
#
# blip_processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
# blip_model = Blip2ForConditionalGeneration.from_pretrained(
#     "Salesforce/blip2-opt-2.7b",
#     torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
# ).to(DEVICE)
#
# captions = []
# for path in tqdm(sample_paths[:10], desc="BLIP-2 captioning"):
#     img = Image.open(path).convert("RGB")
#     inputs = blip_processor(images=img, return_tensors="pt").to(DEVICE, torch.float16)
#     out = blip_model.generate(**inputs, max_new_tokens=80)
#     caption = blip_processor.decode(out[0], skip_special_tokens=True)
#     captions.append({"file": path.name, "caption": caption})
#     print(f"{path.name}: {caption}")

print("BLIP-2 block is commented out — uncomment to enable.")

## 8. Summary Report

In [ ]:
print("=" * 65)
print("PHOTO LIBRARY ANALYSIS — SUMMARY")
print("=" * 65)
print(f"Total images in library : {len(all_paths)}")
print(f"Sample analysed         : {len(sample_paths)}")
print()

print("── Composition ─────────────────────────────────────────────")
ll_mean = df["leading_lines_score"].mean()
rot_mean = df["thirds_score"].mean()
print(f"Avg leading-lines score : {ll_mean:.3f}")
print(f"Avg rule-of-thirds score: {rot_mean:.3f}")
print()

print("── Top 5 images by composition score ───────────────────────")
print(df_sorted[["name", "composition_score", "primary_style"]]
      .head(5).to_string(index=False))
print()

print("── Style distribution ──────────────────────────────────────")
print(df["primary_style"].value_counts().to_string())
print()

print("── Recommendations ─────────────────────────────────────────")
dominant_style = df["primary_style"].value_counts().index[0]
low_rot = df[df["thirds_score"] < 0.05]
high_ll  = df[df["leading_lines_score"] > 0.5]
print(f"Your dominant style     : {dominant_style}")
print(f"Images with weak thirds : {len(low_rot)} ({100*len(low_rot)/len(df):.0f}%)")
print(f"Images with strong lines: {len(high_ll)} ({100*len(high_ll)/len(df):.0f}%)")

if ll_mean > 0.4:
    print("→ Your photos show strong use of leading lines — a clear visual strength.")
if rot_mean < 0.1:
    print("→ Rule-of-thirds alignment is low — consider re-framing subjects off-centre.")
else:
    print("→ Rule-of-thirds usage is healthy across the sample.")

## 9. Export results

In [ ]:
out_csv = Path("photo_analysis_results.csv")
df.drop(columns=["path"]).to_csv(out_csv, index=False)
print(f"Results saved to {out_csv.resolve()}")